In [11]:
import pandas as pd

df_ab = pd.read_csv('/mnt/HC_Volume_18315164/home-jupyter/jupyter-anna-tsoj-syp9958/Проект_1_Задание_2.csv', sep=';', encoding='windows-1251')
df_ab.head()

,user_id,revenue,testgroup
0,1,0,b
1,2,0,a
2,3,0,a
3,4,0,b
4,5,0,b


In [12]:
# Сгруппировала данные по тестовым группам
metrics = df_ab.groupby('testgroup').agg(
    total_users=('user_id', 'count'),                         # Всего пользователей
    paying_users=('revenue', lambda x: (x > 0).sum()),         # Только те, у кого выручка > 0
    total_revenue=('revenue', 'sum')                           # Общая выручка группы
).reset_index()

# Рассчитала метрики на основе агрегированных данных
metrics['CR'] = (metrics['paying_users'] / metrics['total_users'] * 100).round(4) # в %
metrics['ARPU'] = (metrics['total_revenue'] / metrics['total_users']).round(4)
metrics['ARPPU'] = (metrics['total_revenue'] / metrics['paying_users']).round(4)

metrics

,testgroup,total_users,paying_users,total_revenue,CR,ARPU,ARPPU
0,a,202103,1928,5136189,0.9540,25.4137,2663.9984
1,b,202667,1805,5421603,0.8906,26.7513,3003.6582


In [13]:
# Отфильтровала только платящих пользователей для каждой группы
paying_a = df_ab[(df_ab['testgroup'] == 'a') & (df_ab['revenue'] > 0)]['revenue']
paying_b = df_ab[(df_ab['testgroup'] == 'b') & (df_ab['revenue'] > 0)]['revenue']

# Выводим детальное описание распределения платежей
print("Группа А (Контроль) - платящие:")
print(paying_a.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nГруппа B (Тест) - платящие:")
print(paying_b.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

Группа А (Контроль) - платящие:
count     1928.000000
mean      2663.998444
std       9049.039763
min        200.000000
10%        221.000000
25%        257.000000
50%        311.000000
75%        361.000000
90%        393.300000
95%      37299.650000
99%      37340.730000
max      37433.000000
Name: revenue, dtype: float64

Группа B (Тест) - платящие:
count    1805.000000
mean     3003.658172
std       572.619709
min      2000.000000
10%      2202.400000
25%      2513.000000
50%      3022.000000
75%      3478.000000
90%      3795.800000
95%      3891.800000
99%      3981.920000
max      4000.000000
Name: revenue, dtype: float64


In [14]:
from scipy import stats

# Построила таблицу сопряженности: [[платящие_А, неплатящие_А], [платящие_B, неплатящие_B]]
contingency_table = [
    [1928, 202103 - 1928], # Группа А
    [1805, 202667 - 1805]  # Группа B
]

# Провела тест
chi2_stat, p_val, dof, ex = stats.chi2_contingency(contingency_table)
print(f"p-value для конверсии: {p_val:.5f}")

p-value для конверсии: 0.03648


In [19]:
import numpy as np

# Функция для бутстрапа разности средних
def get_bootstrap_diff(data_1, data_2, boot_it=5000):
    boot_data = []
    for i in range(boot_it):
        samples_1 = data_1.sample(len(data_1), replace=True).values
        samples_2 = data_2.sample(len(data_2), replace=True).values
        boot_data.append(np.mean(samples_1) - np.mean(samples_2))
        
    return pd.Series(boot_data)

# Запустила бутстрап для платящих (ARPPU)
boot_results_arppu = get_bootstrap_diff(paying_a, paying_b)

# Нашла 95% доверительный интервал для разницы средних
left_quant = boot_results_arppu.quantile(0.025)
right_quant = boot_results_arppu.quantile(0.975)

print(f"95% Доверительный интервал для разности ARPPU: [{left_quant:.2f}, {right_quant:.2f}]")

95% Доверительный интервал для разности ARPPU: [-741.28, 82.09]


In [20]:
# Вытащила выручку по всем пользователям групп А и B в виде массивов numpy
all_a = df_ab[df_ab['testgroup'] == 'a']['revenue'].values
all_b = df_ab[df_ab['testgroup'] == 'b']['revenue'].values

# Быстрый бутстрап разности средних (ARPU)
boot_data_arpu = []
for i in range(1000):
    samples_a = np.random.choice(all_a, size=len(all_a), replace=True)
    samples_b = np.random.choice(all_b, size=len(all_b), replace=True)
    boot_data_arpu.append(np.mean(samples_a) - np.mean(samples_b))

boot_results_arpu = pd.Series(boot_data_arpu)

# Посчитала доверительный интервал
left_quant_arpu = boot_results_arpu.quantile(0.025)
right_quant_arpu = boot_results_arpu.quantile(0.975)

print(f"95% Доверительный интервал для разности ARPU: [{left_quant_arpu:.2f}, {right_quant_arpu:.2f}]")

95% Доверительный интервал для разности ARPU: [-5.44, 2.95]


In [22]:
# Статистические результаты:

# Конверсия (CR): Значимо снизилась в тестовой группе B (p-value = 0.036).
# Средний чек платящих (ARPPU): Статистически значимых различий нет (95% доверительный интервал разницы включает 0: [-741.28, 82.09]).
# Средний доход на пользователя (ARPU): Статистически значимых различий также нет (95% доверительный интервал разницы включает 0: [-5.44, 2.95]).

# Продуктовая интерпретация:

# В контрольной группе А выручка держится на «китах» (около 5% платящих пользователей совершают огромные покупки по ~37 000 руб., в то время как остальные платят по 200–400 руб.). Это классическая, но рискованная геймдев-модель: если уйдёт пара крупных игроков, экономика просядет.
# В тестовой группе B пользователям предложили более дорогой пакет (от 2000 до 4000 руб.). Это отсекло мелких плательщиков (конверсия упала), но заставило платить средний сегмент. Дисперсия здесь минимальна, доходы более предсказуемы.
# Однако, поскольку тест не показал статистически значимого прироста в деньгах на одного пользователя (ARPU не изменился значимо), раскатывать вариант B на всех пользователей в текущем виде не рекомендуется.